In [ ]:
# Cell 1: Environment Setup

include("helpers/initialization_helpers.jl")
include("helpers/initialization_dictionaries.jl")
using .InitializationHelpers
using .InitializationDictionaries
using OMJulia
using Plots, DataFrames, CSV

# --- Configuration ---

# 1. Directory containing the model files
MODEL_DIR = abspath("models")

# 2. Select the model to initialize
MODEL = "BESSloadAB"

# 3. Path to the selected dynamic case file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Name and path of the initialized dynamic case file
INITIALIZED_MODEL = MODEL * "_initialized"
INITIALIZED_FILE_PATH = joinpath(MODEL_DIR, INITIALIZED_MODEL * ".mo")

# 5. Path to the auxiliary file
AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE_PATH = joinpath(MODEL_DIR, AUX_MODEL * ".mo")

# 6. Path to the Dynawo package.mo
DYNAWO_PKG_PATH = "/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 7. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 8. Variable to plot after the initialized simulation
# PLOT_VARIABLE = "generatorSynchronous.terminal.i.re"  # SMIB option
# PLOT_VARIABLE = "BESS.terminal.V.im"  # MyBESS option
PLOT_VARIABLE = "BESS.measurements.PPu"


In [ ]:
# Cell 2: OpenModelica Setup + Model Loading

# 1. Load the dynamic model
BESS = OMJulia.OMCSession()
sendExpression(BESS, "loadModel(Complex)")
sendExpression(BESS, "loadModel(ModelicaServices)")
sendExpression(BESS, "loadFile(\"$MODELICA_PKG_PATH\")")
sendExpression(BESS, "loadFile(\"$DYNAWO_PKG_PATH\")")
sendExpression(BESS, "loadFile(\"$MODEL_FILE_PATH\")")
sendExpression(BESS, "clearMessages()")
println("Checking the dynamic model...")
chk_dyn = sendExpression(BESS, "checkModel($MODEL)", parsed=false)
println(chk_dyn)

# 2. Load the auxiliary model
StaticBESS = OMJulia.OMCSession()
sendExpression(StaticBESS, "loadModel(Complex)")
sendExpression(StaticBESS, "loadModel(ModelicaServices)")
sendExpression(StaticBESS, "loadFile(\"$MODELICA_PKG_PATH\")")
sendExpression(StaticBESS, "loadFile(\"$DYNAWO_PKG_PATH\")")
sendExpression(StaticBESS, "loadFile(\"$AUX_FILE_PATH\")")
sendExpression(StaticBESS, "clearMessages()")
println("Checking the auxiliary model...")
chk_aux = sendExpression(StaticBESS, "checkModel($AUX_MODEL)", parsed=false)
println(chk_aux)


In [ ]:
# Cell 3: Simulate the auxiliary model and extract initialization values
# 1. Build and simulate the auxiliary model
ModelicaSystem(StaticBESS, AUX_FILE_PATH, AUX_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
simulate(StaticBESS, resultfile = AUX_MODEL * "_res.mat")

# 2. Get the dynamic model components that need initialization
components = get_all_components(BESS, MODEL)
initializable_components = get_initializable_components(components, INIT_PARAMS)

# 3. Extract initialization values from the auxiliary model
init_values_by_component = extract_all_initialization_values(StaticBESS, initializable_components, INIT_PARAMS)


In [ ]:
# Cell 4: Build and save the initialized dynamic model
om_send(BESS, "deleteClass($INITIALIZED_MODEL)")
om_send(BESS, "clearMessages()")
om_send(BESS, "copyClass($MODEL, \"$INITIALIZED_MODEL\")")

apply_initialization_modifiers!(BESS, INITIALIZED_MODEL, initializable_components, INIT_PARAMS, init_values_by_component)

om_send(BESS, "saveModel(\"$INITIALIZED_FILE_PATH\", $INITIALIZED_MODEL)")
println("Wrote initialized model: ", INITIALIZED_FILE_PATH)


In [ ]:
# Cell 5: Final simulation
InitializedBESS = OMJulia.OMCSession()
sendExpression(InitializedBESS, "loadModel(Complex)")
sendExpression(InitializedBESS, "loadModel(ModelicaServices)")
ModelicaSystem(InitializedBESS, INITIALIZED_FILE_PATH, INITIALIZED_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
sendExpression(InitializedBESS, "clearMessages()")
println("Checking the initialized model...")
chk_init = sendExpression(InitializedBESS, "checkModel($INITIALIZED_MODEL)", parsed=false)
println(chk_init)

initialized_resultfile_name = INITIALIZED_MODEL * ".csv"
simulate(InitializedBESS, resultfile = initialized_resultfile_name, simflags = "-outputFormat=csv")
initialized_resultfile = joinpath(getWorkDirectory(InitializedBESS), initialized_resultfile_name)


In [ ]:
# Cell 6: Plot the initialized dynamic model
initialized_df = DataFrame(CSV.File(initialized_resultfile))

plotlyjs()
p = plot(initialized_df[!, "time"], initialized_df[!, PLOT_VARIABLE], label = [PLOT_VARIABLE])
plot!(p, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p, "Initialized dynamic model response")
xlabel!(p, "Time (s)")
ylabel!(p, PLOT_VARIABLE)
